# Week 8 (Tue) - Class Activities in CS 315

You are given the file `netflix_top10_countries.tsv` in the folder data, solve the following tasks using it.

**Table of Contents**

1. [Q1: Load the file](#q1)
2. [Q2: Show number of rows](#q2)
3. [Q3: Show column names](#q3)
4. [Q4: Show number of total countries](#q4)
5. [Q5: Show number of TV series](#q5)
6. [Q6: Number of TV series in South Korea](#q6)
7. [Q7: Total weeks of Squid Games in Italy ](#q7)
8. [Q8: Time frame for a given country](#q8)
9. [Q9: Is number of weeks in ranking equal?](#q9)
10. [Q10: Country with fewest weeks](#q10)
11. [Q11: Rank to score](#q11)


<a id="q1"></a>
### Q1: Load the file

In [2]:
import pandas as pd

In [3]:
#df = pd.read_csv("netflix_top10_countries.tsv", sep="\t")
df = pd.read_csv("all-weeks-countries.tsv", sep="\t")

In [4]:
df.head()

,country_name,country_iso2,week,category,weekly_rank,show_title,season_title,cumulative_weeks_in_top_10
0,Argentina,AR,2026-10-18,Films,1,Therapy Done Right,NaN,1
1,Argentina,AR,2026-10-18,Films,2,Animals,NaN,2
2,Argentina,AR,2026-10-18,Films,3,Anyone But You,NaN,2
3,Argentina,AR,2026-10-18,Films,4,Doing Life,NaN,3
4,Argentina,AR,2026-10-18,Films,5,Ice Age: Collision Course,NaN,5


<a id="q2"></a>
### Q2: Show number of rows

In [5]:
df.shape[0] #rows

515920

<a id="q3"></a>
### Q3: Show column names

In [6]:
df.columns #show col names

Index(['country_name', 'country_iso2', 'week', 'category', 'weekly_rank',
       'show_title', 'season_title', 'cumulative_weeks_in_top_10'],
      dtype='object')

<a id="q4"></a>
### Q4: Show number of total countries

In [7]:
df["country_name"].nunique() #get num of unique countries

94

<a id="q5"></a>
### Q5: Show number of TV series

In [ ]:
df["show_title"][df["category"]== "TV"].nunique() #3574 unique shows; updated 3607

3607

<a id="q6"></a>
### Q6: Number of TV series in South Korea

In [ ]:
df["show_title"][(df["country_name"]=="South Korea") & (df["category"] == "TV")].nunique() #481; updated 486

486

<a id="q7"></a>
### Q7: Total weeks of Squid Game in Italy

In [10]:
df["week"][(df["country_name"]=="Italy") & (df["show_title"]=="Squid Game")].nunique()


29

<a id="q8"></a>
### Q8: Time frame for a given country

Write a function that given a country name prints out the start and end date of their rankings.

Once you have created the function, call it for a few countries to test it.

In [ ]:
def show_timeframe(df, country):
    '''Prints out the start and end date of rankings for a country'''
    country_df = df[df["country_name"] == country]

    if country_df.empty: #no data
        print(f"No Netflix rankings found for {country}.")
    else:
        start_date = country_df["week"].min()
        end_date = country_df["week"].max()
        print(f"Netflix rankings for {country} exist between {start_date} and {end_date}")

In [16]:
show_timeframe(df, "Chile")
show_timeframe(df, "Russia")
show_timeframe(df, "Haiti")

Netflix rankings for Chile exist between 2021-07-04 and 2026-10-18
Netflix rankings for Russia exist between 2021-07-04 and 2022-02-27
No Netflix rankings found for Haiti.


<a id="q9"></a>
### Q9: Is number of weeks equal for each country?

The total number of weeks in the rankings is `len(df['week'].unique())`.

The total number of countries is `len(df['country_name'].unique())`.  

For every week, each country has 20 rows.   

If they all stayed the same number of weeks, then the product of these three weeks would be the same as the total number of rows.  

In [17]:
#boolean expression
len(df) ==  len(df["week"].unique()) * len(df['country_name'].unique()) * 20

False

We should have expected this to be `False` since we saw in Q8 that Russia stopped being in the rankings after it attacked Ukraine in February 2022.

<a id="q10"></a>
### Q10: Country with fewest weeks

Write a one-liner that will show the country with the fewest weeks in the Netflix rankings

Need to use .groupby by countries to get all weeks for each, then find the country with the smallest number of weeks.

In [23]:
df.groupby("country_name")['week'].nunique().idxmin() #finds num weeks by country and uses .idxmin() to get country with min num 

'Russia'

<a id="q11"></a>
### Q11: Rank to score

Need to create a dict to match the ranks to a score in F1 scoring system

In [28]:
def rank2score(rank):
    '''Takes in a rank and returns a score based on the F1 scoring system'''
    #create a dict with key-value pairs of rank:score
    point_system = {1: 25,
                    2: 18,
                    3: 15, 
                    4: 12, 
                    5: 10, 
                    6: 8, 
                    7: 6,
                    8: 4, 
                    9: 2, 
                    10: 1
                    }
    return point_system.get(rank)

In [33]:
rank2score(6)

8

Use df.apply() to apply the function to the weekly rank col to turn to scores

In [62]:
df["score"] = df['weekly_rank'].apply(rank2score)

df.drop(columns = ["season_title", "country_iso2", "category"]).head(10)

,country_name,week,weekly_rank,show_title,cumulative_weeks_in_top_10,score
0,Argentina,2026-10-18,1,Therapy Done Right,1,25
1,Argentina,2026-10-18,2,Animals,2,18
2,Argentina,2026-10-18,3,Anyone But You,2,15
3,Argentina,2026-10-18,4,Doing Life,3,12
4,Argentina,2026-10-18,5,Ice Age: Collision Course,5,10
5,Argentina,2026-10-18,6,Corpse Bride,1,8
6,Argentina,2026-10-18,7,Acid,2,6
7,Argentina,2026-10-18,8,Ice Age: Dawn of the Dinosaurs,6,4
8,Argentina,2026-10-18,9,The Storm,1,2
9,Argentina,2026-10-18,10,The Equalizer 3,3,1


## Kdramas for Final Project

To get the df shown in the tasks, we need to group by country_name and aggregate the score, weeks, and rankings:

In [58]:
lovely_runner = df[df["show_title"] == "Lovely Runner"]
lovely_runner.groupby("country_name").agg(total_score=("score", "sum"),
                                          total_weeks=("week", "nunique"),
                                          rankings_list=("weekly_rank", list) #turn all rankings into list format
                                          ).reset_index().sort_values("total_score", ascending=False)
# need .reset_index() first to maintain index order after sorting

,country_name,total_score,total_weeks,rankings_list
12,Vietnam,75,7,"[8, 7, 6, 5, 4, 1, 5]"
2,Indonesia,69,7,"[9, 7, 6, 7, 4, 1, 5]"
3,Malaysia,62,6,"[9, 9, 7, 3, 1, 4]"
6,Philippines,41,4,"[7, 5, 3, 5]"
4,Maldives,32,4,"[10, 6, 3, 6]"
9,Sri Lanka,32,4,"[10, 7, 3, 5]"
0,Bangladesh,30,3,"[5, 4, 6]"
1,India,26,3,"[8, 4, 5]"
7,Singapore,20,4,"[9, 7, 5, 9]"
11,Thailand,20,3,"[7, 8, 5]"


### For Kdramas related to beauty standards:
- True Beauty
- My ID is Gangnam Beauty

In [59]:
true_beauty = df[df["show_title"] == "True Beauty"]
true_beauty.groupby("country_name").agg(total_score=("score", "sum"),
                                          total_weeks=("week", "nunique"),
                                          rankings_list=("weekly_rank", list) #turn all rankings into list format
                                          ).reset_index().sort_values("total_score", ascending=False)
# need .reset_index() first to maintain index order after sorting

,country_name,total_score,total_weeks,rankings_list
3,Bangladesh,215,18,"[9, 8, 6, 6, 5, 5, 5, 6, 5, 4, 4, 5, 5, 6, 1, ..."
4,Bolivia,191,25,"[10, 10, 8, 9, 10, 6, 8, 10, 10, 8, 9, 7, 8, 5..."
38,Philippines,180,14,"[10, 9, 6, 5, 7, 3, 4, 5, 3, 3, 2, 2, 1, 1]"
41,Saudi Arabia,178,17,"[10, 8, 9, 9, 10, 8, 8, 4, 4, 5, 5, 4, 2, 1, 1..."
33,Oman,178,18,"[10, 10, 8, 9, 8, 9, 10, 8, 4, 5, 6, 7, 6, 3, ..."
37,Peru,163,18,"[8, 9, 10, 10, 9, 10, 5, 7, 6, 7, 4, 5, 6, 3, ..."
24,Maldives,153,15,"[10, 9, 9, 7, 8, 6, 3, 6, 6, 6, 6, 3, 2, 1, 1]"
23,Malaysia,142,14,"[10, 8, 7, 9, 8, 6, 6, 5, 6, 6, 3, 2, 1, 1]"
10,Ecuador,138,17,"[10, 10, 9, 9, 10, 6, 7, 6, 8, 5, 6, 8, 6, 5, ..."
39,Qatar,137,12,"[10, 8, 7, 5, 6, 6, 7, 6, 2, 2, 1, 1]"


In [60]:
id_gangnam = df[df["show_title"] == "My ID is Gangnam Beauty"]
id_gangnam.groupby("country_name").agg(total_score=("score", "sum"),
                                          total_weeks=("week", "nunique"),
                                          rankings_list=("weekly_rank", list) #turn all rankings into list format
                                          ).reset_index().sort_values("total_score", ascending=False)
# need .reset_index() first to maintain index order after sorting

,country_name,total_score,total_weeks,rankings_list
3,Hong Kong,30,3,"[7, 4, 4]"
8,Peru,16,3,"[8, 7, 7]"
7,Nicaragua,12,3,"[8, 8, 8]"
10,Taiwan,10,2,"[9, 6]"
5,Malaysia,9,3,"[9, 7, 10]"
1,El Salvador,6,2,"[8, 9]"
2,Honduras,5,3,"[10, 9, 9]"
9,Philippines,2,1,[9]
0,Chile,1,1,[10]
4,Indonesia,1,1,[10]
